# Public portfolio adaptation
Adapted from my Fall 2025 Gilead project. Outputs and original data are excluded.
Company examples are fictional. Requires your own Databricks workspace and credentials.
This notebook is retained as research code; see the repository documentation for limitations.


# Final Pipeline: Rating Script
## Models: Llama -> FinBERT -> Claude
## Window Size: 5 (2 Previous, Current, 2 Next)

In [ ]:
# Install required packages
%pip install anthropic transformers torch accelerate mlflow python-dotenv
%restart_python

In [ ]:
import anthropic
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from mlflow.deployments import get_deploy_client
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
import os
from dotenv import load_dotenv
import math

spark = SparkSession.builder.getOrCreate()

In [ ]:
# ==================================================
# CONFIGURATION
# ==================================================

# Models
FINBERT_MODEL = "ProsusAI/finbert"
LLAMA_ENDPOINT = "databricks-meta-llama-3-1-8b-instruct"
CLAUDE_MODEL = "claude-sonnet-4-20250514" # Verify model name availability

# API Keys
# Ensure .env file exists or set environment variables
load_dotenv()
CLAUDE_API_KEY = os.environ.get("CLAUDE_API_KEY")
if not CLAUDE_API_KEY:
    print("WARNING: CLAUDE_API_KEY not found. Claude rating will fail if not set.")

# Initialize Clients
try:
    db_client = get_deploy_client("databricks")
except Exception as e:
    print(f"Warning: Could not connect to Databricks serving client: {e}")

try:
    claude_client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
except Exception as e:
    print(f"Warning: Could not initialize Claude client: {e}")

In [ ]:
# ==================================================
# HELPER FUNCTIONS
# ==================================================

def polarity_to_1_7(p_pos: float, p_neg: float) -> float:
    """Convert polarity scores to 1-7 scale"""
    return float(max(1.0, min(7.0, 4.0 + 3.0 * (p_pos - p_neg))))

def round_to_sig_figs(value: float, sig_figs: int = 2) -> float:
    """Round a number to specified significant figures"""
    if value == 0.0:
        return 0.0
    return round(value, sig_figs - int(math.floor(math.log10(abs(value)))) - 1)

def extract_json(text):
    """Robust JSON extraction from model responses"""
    if text is None or not isinstance(text, str):
        return {"score": 4}
    
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "score" in parsed:
            return parsed
    except:
        pass
    
    match = re.search(r'\{[^}]*"score"[^}]*\}', text)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, dict):
                return parsed
        except:
            pass
    
    number_match = re.search(r'"score":\s*([0-9.]+)', text)
    if number_match:
        try:
            score_val = float(number_match.group(1))
            return {"score": score_val}
        except:
            pass
    
    return {"score": 4}

In [ ]:
# ==================================================
# DATA PREPARATION
# ==================================================

def prepare_data(df):
    print("Preparing data and creating windows...")
    
    # Ensure columns exist
    required_columns = ["Global #", "Doc #", "Sentence #", "Context", "Firm name", "Year", "Quarter", "File title", "Doc published date"]
    for col in required_columns:
        if col not in df.columns:
            print(f"Error: Missing column '{col}' in dataframe")
            return None

    # Standardize column names
    df = df.withColumnRenamed("Global #", "global_number") \
           .withColumnRenamed("Doc #", "report_id") \
           .withColumnRenamed("Sentence #", "sentence_number") \
           .withColumnRenamed("Firm name", "firm_name") \
           .withColumnRenamed("Context", "sentence") \
           .withColumnRenamed("File title", "file_title") \
           .withColumnRenamed("Doc published date", "doc_date") \
           .withColumnRenamed("Year", "year") \
           .withColumnRenamed("Quarter", "quarter")

    # Create Window for 5-chunk context (2 Previous, Current, 2 Next)
    w = Window.partitionBy("report_id").orderBy("sentence_number")
    
    windowed = (df
        .withColumn("sentence_n_minus2", F.lag("sentence", 2).over(w))
        .withColumn("sentence_n_minus1", F.lag("sentence", 1).over(w))
        .withColumn("sentence_n", F.col("sentence"))
        .withColumn("sentence_n_plus1", F.lead("sentence", 1).over(w))
        .withColumn("sentence_n_plus2", F.lead("sentence", 2).over(w))
    )

    # Construct 5-chunk text
    # MODIFIED: Use concat_ws to handle missing neighbors gracefully (partial context)
    final_df = windowed.select(
        "global_number", "report_id", "sentence_number", "firm_name", "year", "quarter", "file_title", "doc_date",
        F.lit("5_chunk").alias("window_type"),
        F.concat_ws(" ", 
            F.col("sentence_n_minus2"),
            F.col("sentence_n_minus1"),
            F.col("sentence_n"),
            F.col("sentence_n_plus1"),
            F.col("sentence_n_plus2")
        ).alias("window_text")
    ).filter(F.col("sentence_n").isNotNull())
    
    return final_df

In [ ]:
# ==================================================
# 1. LLAMA SCORING
# ==================================================

LLAMA_PROMPT = (
    'Financial sentiment analysis on {firm_name} stock performance. '
    'Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on {firm_name}. '
    'Return ONLY strict JSON: {{\"id\":\"<id>\",\"score\":<decimal>}}. '
    'Scores must be decimal numbers ranging from 0.0 to 7.0. '
    'Scale: 0.0=objective (NON-commentary), 1.0=very pessimistic, 4.0=neutral commentary, 7.0=very optimistic. '
    'Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary. '
    'Text: '
)

def score_llama(row):
    if row.window_text is None:
        return {**row.asDict(), "llama_score": None}
    
    try:
        prompt = LLAMA_PROMPT.format(firm_name=row.firm_name)
        full_prompt = prompt + f"[{row.report_id}|{row.sentence_number}]: {row.window_text[:2000]}"
        
        response = db_client.predict(
            endpoint=LLAMA_ENDPOINT,
            inputs={
                "messages": [{"role": "user", "content": full_prompt}],
                "temperature": 0.0,
                "max_tokens": 30
            }
        )
        raw_content = response["choices"][0]["message"]["content"]
        result = extract_json(raw_content)
        score = float(result.get("score", 4))
        
        if not (0 <= score <= 7):
            score = "Error"
        else:
            score = round_to_sig_figs(score, 2)
            
        return {**row.asDict(), "llama_score": score}
        
    except Exception as e:
        # print(f"Llama error for {row.report_id}-{row.sentence_number}: {e}")
        return {**row.asDict(), "llama_score": 4} # Default to neutral on error

def run_llama(df):
    print("Running Llama...")
    rows = df.collect()
    total_rows = len(rows)
    results = []
    
    # Process in batches to show progress
    batch_size = 20
    
    for i in range(0, total_rows, batch_size):
        batch = rows[i:i+batch_size]
        
        with ThreadPoolExecutor(max_workers=3) as executor:
            futures = [executor.submit(score_llama, row) for row in batch]
            for future in as_completed(futures):
                results.append(future.result())
        
        if (i + len(batch)) % 100 == 0 or (i + len(batch)) == total_rows:
            print(f"Llama: Processed {i + len(batch)}/{total_rows} rows ({100*(i + len(batch))/total_rows:.1f}%)")
            
    return spark.createDataFrame(results)

In [ ]:
# ==================================================
# 2. FINBERT SCORING
# ==================================================

def run_finbert(df):
    print("Running FinBERT...")
    
    # Load model
    device = 0 if torch.cuda.is_available() else -1
    tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
    pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, top_k=None, device=device)
    
    rows = df.collect()
    total_rows = len(rows)
    texts = [r.window_text or "" for r in rows]
    
    # Batch processing
    batch_size = 32
    results = []
    
    for i in range(0, total_rows, batch_size):
        batch_rows = rows[i:i+batch_size]
        batch_texts = texts[i:i+batch_size]
        
        try:
            outs = pipe(batch_texts, truncation=True, max_length=512)
            for r, scores in zip(batch_rows, outs):
                if r.window_text is None:
                    score = None
                else:
                    probs = {s["label"].lower(): float(s["score"]) for s in scores}
                    p_pos = probs.get("positive", 0.0)
                    p_neg = probs.get("negative", 0.0)
                    score = polarity_to_1_7(p_pos, p_neg)
                    score = round_to_sig_figs(score, 2)
                
                res_dict = r.asDict()
                res_dict["finbert_score"] = score
                results.append(res_dict)
        except Exception as e:
            print(f"FinBERT batch error: {e}")
            for r in batch_rows:
                res_dict = r.asDict()
                res_dict["finbert_score"] = "Error"
                results.append(res_dict)
        
        if (i + len(batch_rows)) % 100 == 0 or (i + len(batch_rows)) == total_rows:
            print(f"FinBERT: Processed {i + len(batch_rows)}/{total_rows} rows ({100*(i + len(batch_rows))/total_rows:.1f}%)")
                
    return spark.createDataFrame(results)

In [ ]:
# ==================================================
# 3. CLAUDE SCORING
# ==================================================

CLAUDE_PROMPT_TEMPLATE = (
    "You are rating Wall Street financial analysis on {firm_name} stock performance. "
    "Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on {firm_name}. "
    "Return ONLY a valid JSON array, where each item is {{\"id\":\"<id>\",\"score\":<decimal>}}.\n"
    "Scores must be decimal numbers with 2 significant figures, ranging from 0.0 to 7.0.\n"
    "Scale: 0.0 = objective (NON-commentary), 1.0 = very pessimistic, 4.0 = neutral commentary, 7.0 = very optimistic.\n"
    "Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary.\n"
    "Items:\n{items}\n"
    "Respond with ONLY the JSON array."
)

def run_claude(df):
    print("Running Claude...")
    rows = df.collect()
    valid_rows = [r for r in rows if r.window_text is not None]
    total_valid = len(valid_rows)
    
    valid_rows.sort(key=lambda x: x.firm_name)
    
    results_map = {}
    batch_size = 5
    
    for i in range(0, total_valid, batch_size):
        batch = valid_rows[i:i+batch_size]
        current_firm = batch[0].firm_name
        
        items_list = []
        for row in batch:
            text = row.window_text[:1500]
            items_list.append(f"[{row.report_id}|{row.sentence_number}]: {text}")
            
        items_str = "\n".join(items_list)
        prompt = CLAUDE_PROMPT_TEMPLATE.format(firm_name=current_firm, items=items_str)
        
        try:
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=500,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text.strip()
            
            # Parse JSON
            try:
                json_data = json.loads(response_text)
            except:
                # Try regex
                match = re.search(r'\[.*\]', response_text, re.DOTALL)
                if match:
                    json_data = json.loads(match.group())
                else:
                    json_data = []
            
            if isinstance(json_data, list):
                for item in json_data:
                    if "id" in item and "score" in item:
                        item_id = item["id"]
                        score = float(item["score"])
                        score = round_to_sig_figs(score, 2)
                        results_map[item_id] = score
                        
        except Exception as e:
            print(f"Claude batch error: {e}")
            
        if (i + len(batch)) % 20 == 0 or (i + len(batch)) == total_valid:
            print(f"Claude: Processed {i + len(batch)}/{total_valid} valid rows ({100*(i + len(batch))/total_valid:.1f}%)")
            
    # Merge results back
    final_results = []
    for row in rows:
        res_dict = row.asDict()
        if row.window_text is None:
            res_dict["claude_score"] = None
        else:
            key = f"{row.report_id}|{row.sentence_number}"
            res_dict["claude_score"] = results_map.get(key, "Error")
        final_results.append(res_dict)
        
    return spark.createDataFrame(final_results)

In [ ]:
# ==================================================
# MAIN EXECUTION
# ==================================================

# 1. Load Data
print("Loading data from parsed_data table...")
try:
    raw_df = spark.table("parsed_data")
    
    # Cast columns
    raw_df = raw_df.withColumn("Global #", F.col("Global #").cast("int")) \
                   .withColumn("Doc #", F.col("Doc #").cast("int")) \
                   .withColumn("Sentence #", F.col("Sentence #").cast("int"))
                   
    df = prepare_data(raw_df)
except Exception as e:
    print(f"Error loading table: {e}")
    df = None

if df is not None:
    # 2. Run Llama
    df_llama = run_llama(df)
    
    # 3. Run FinBERT
    df_finbert = run_finbert(df_llama)
    
    # 4. Run Claude
    final_df = run_claude(df_finbert)
    
    # Sort by Global #
    final_df = final_df.orderBy("global_number")
    
    # 5. Save Results
    output_table = "final_pipeline_results"
    final_df.write.mode("overwrite").saveAsTable(output_table)
    print(f"✅ Pipeline complete. Results saved to table '{output_table}'")
    display(final_df)

In [ ]:
# ==================================================
# VALIDATION
# ==================================================
print("\nValidation:")
input_count = raw_df.count()
output_count = final_df.count()
print(f"Input rows: {input_count}")
print(f"Output rows: {output_count}")

if input_count == output_count:
    print("✅ Success: No lines were missed.")
else:
    print(f"⚠️ Warning: {input_count - output_count} lines were dropped.")